In [0]:
%sql
update pricing_analytics.silver.daily_pricing set
PRODUCTGROUP_NAME='Ground_vegitable',
LAKE_HOUSE_UPDATED_DATE=current_timestamp()
where PRODUCT_NAME='Onion'

In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype1_dim1
select distinct PRODUCT_NAME,
PRODUCTGROUP_NAME,
current_timestamp() as lake_house_updated_date,
current_timestamp() as lakehouse_inserted_date
from pricing_analytics.silver.daily_pricing where LAKE_HOUSE_UPDATED_DATE>(select coalesce(max(process_table_date_time), date('1900-01-01')) from pricing_analytics.bronze.pipeline_control_logs where process_name='pricing_data_silver_product_scdtype1' and process_status='Completed')

In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype1_dim2
select 
silverDim.PRODUCT_NAME,
silverDim.PRODUCTGROUP_NAME,
goldDim.PRODUCT_NAME as GOLD_PRODUCT_NAME,
case when goldDim.PRODUCT_NAME is null then
row_number() over(order by silverDim.PRODUCT_NAME, silverDim.PRODUCTGROUP_NAME) else goldDim.PRODUCT_ID end as PRODUCT_ID,
current_timestamp() as lake_house_updated_date,
current_timestamp() as lakehouse_inserted_date
from pricing_analytics.silver.productscdtype1_dim1 silverDim  
left join pricing_analytics.gold.product_scdtype1_dim goldDim
on silverDim.PRODUCT_NAME=goldDim.PRODUCT_NAME
where goldDim.PRODUCT_NAME is null or silverDim.PRODUCTGROUP_NAME<>goldDim.PRODUCTGROUP_NAME

In [0]:
%sql
create or replace table pricing_analytics.silver.productscdtype1_dim3
select
PRODUCT_NAME,
PRODUCTGROUP_NAME,
case when GOLD_PRODUCT_NAME is null then 
(PRODUCT_ID+PREV_MAX_ID) else PRODUCT_ID end as PRODUCT_ID,
case when GOLD_PRODUCT_NAME is null then 'NEW' else 'UPDATED' end as RECORD_STATUS,
current_timestamp() as lake_house_inserted_date,
current_timestamp() as lake_house_updated_date
from pricing_analytics.silver.productscdtype1_dim2
cross join (select coalesce(max(PRODUCT_ID), 0) as PREV_MAX_ID from pricing_analytics.gold.product_scdtype1_dim)

In [0]:
%sql
merge into pricing_analytics.gold.product_scdtype1_dim goldDim
using pricing_analytics.silver.productscdtype1_dim3 silverDim
on goldDim.PRODUCT_ID=silverDim.PRODUCT_ID

when matched then update set
goldDim.PRODUCTGROUP_NAME=silverDim.PRODUCTGROUP_NAME,
goldDim.RECORD_STATUS='CHANGED',
lake_house_updated_date=current_timestamp()

when not matched then
insert *

In [0]:
%sql
insert into pricing_analytics.bronze.pipeline_control_logs(
  select 
  'pricing_data_silver_product_scdtype1',
  max(LAKE_HOUSE_UPDATED_DATE),
  'Completed',
  current_timestamp()
  from pricing_analytics.silver.daily_pricing
)